In [0]:
-- TODO: manual review required — choose the BTEQ execution wrapper: Databricks notebook, stored procedure, or Workflows DAG.
-- TODO: manual review required — confirm whether former BTEQ ERRORCODE exits should rely on fail-fast execution or explicit Workflows branches.
-- assumption: the Databricks job selects the environment-specific catalog, so source <DB_ENV> prefixes are omitted from schema names.

/**************************************************************************\
* TAKE A TIMESTAMP SNAPSHOT                                                *
\**************************************************************************/

DELETE FROM DB_TRANS_IMPORTS.AML_TMS_SANCSYS;

-- Converted CURRENT_TIMESTAMP(0) to second precision with timezone-naive storage semantics.
INSERT INTO DB_TRANS_IMPORTS.AML_TMS_SANCSYS
(
       SSQ_ID,
       TMS
)
VALUES
(
       1,
       CAST(DATE_TRUNC('SECOND', CURRENT_TIMESTAMP) AS TIMESTAMP_NTZ)
);

/**************************************************************************\
* REMOVE HEADING AND NON NUMERIC GROUP ROWS                                *
\**************************************************************************/

-- Converted OTRANSLATE to TRANSLATE.
DELETE FROM DB_TRANS_IMPORTS.AML_SANCSYS_RAW
WHERE TRIM(COALESCE(GROUP_ID, '')) = ''
   OR TRIM(TRANSLATE(GROUP_ID, '0123456789', '          ')) <> '';

/**************************************************************************\
* TRUNCATED FILE PROTECTION                                                *
\**************************************************************************/

-- Converted the BTEQ ACTIVITYCOUNT exit to an explicit SQL-scripting assertion.
BEGIN
  IF EXISTS (
    SELECT 1
    FROM (SELECT COUNT(DISTINCT GROUP_ID) AS STG_CNT
          FROM DB_TRANS_IMPORTS.AML_SANCSYS_RAW) STG
    CROSS JOIN (SELECT COUNT(*) AS ACT_CNT
                FROM DB_ODS.SANC_ENTITY
                WHERE ACTIVE_IND = 'Y') ACT
    WHERE STG.STG_CNT < ACT.ACT_CNT * 0.5
  ) THEN
    SIGNAL SQLSTATE '45000'
      SET MESSAGE_TEXT = 'Sanctions source appears truncated: staged designations are below half of active designations';
  END IF;
END;

/**************************************************************************\
* POPULATE SANC_ENTITY_STG                                                 *
\**************************************************************************/

DELETE FROM DB_TRANS_IMPORTS.SANC_ENTITY_STG;

-- Converted Teradata formatted DATE casts to TO_DATE with Databricks format tokens.
INSERT INTO DB_TRANS_IMPORTS.SANC_ENTITY_STG
(
       GROUP_ID,
       GROUP_TYP_CD,
       REGIME_NM,
       LISTED_DT,
       LAST_UPD_DT,
       UK_SANC_REF,
       BIRTH_DT,
       NATIONALITY_CTRY_NM,
       NATIONALITY_CTRY_CD,
       ADDR_CTRY_NM,
       ADDR_CTRY_CD,
       PASSPORT_NR
)
SELECT CAST(TRIM(RAW.GROUP_ID) AS INT),
       CASE UPPER(TRIM(RAW.GROUP_TYPE))
            WHEN 'INDIVIDUAL' THEN 'I'
            WHEN 'ENTITY'     THEN 'E'
            WHEN 'SHIP'       THEN 'S'
            ELSE 'E'
       END,
       NULLIF(TRIM(RAW.REGIME), ''),
       CASE WHEN TRIM(RAW.LISTED_ON) LIKE '__/__/____'
                  AND SUBSTR(TRIM(RAW.LISTED_ON), 1, 2) <> '00'
                  AND SUBSTR(TRIM(RAW.LISTED_ON), 4, 2) <> '00'
            THEN TO_DATE(TRIM(RAW.LISTED_ON), 'dd/MM/yyyy')
            ELSE NULL
       END,
       CASE WHEN TRIM(RAW.LAST_UPDATED) LIKE '__/__/____'
                  AND SUBSTR(TRIM(RAW.LAST_UPDATED), 1, 2) <> '00'
                  AND SUBSTR(TRIM(RAW.LAST_UPDATED), 4, 2) <> '00'
            THEN TO_DATE(TRIM(RAW.LAST_UPDATED), 'dd/MM/yyyy')
            ELSE NULL
       END,
       NULLIF(TRIM(RAW.UK_SANC_LIST_REF), ''),
       CASE WHEN TRIM(RAW.DOB) LIKE '__/__/____'
                  AND SUBSTR(TRIM(RAW.DOB), 1, 2) <> '00'
                  AND SUBSTR(TRIM(RAW.DOB), 4, 2) <> '00'
            THEN TO_DATE(TRIM(RAW.DOB), 'dd/MM/yyyy')
            ELSE NULL
       END,
       NULLIF(TRIM(RAW.NATIONALITY), ''),
       NAT.CTRY_2_CD,
       NULLIF(TRIM(RAW.ADDR_CTRY), ''),
       ADR.CTRY_2_CD,
       NULLIF(TRIM(RAW.PASSPORT_NR), '')
FROM DB_TRANS_IMPORTS.AML_SANCSYS_RAW RAW
LEFT JOIN DB_REFERENCE.CTRY_NAME_XREF NAT
  ON NAT.CTRY_NM_TXT = UPPER(TRIM(RAW.NATIONALITY))
LEFT JOIN DB_REFERENCE.CTRY_NAME_XREF ADR
  ON ADR.CTRY_NM_TXT = UPPER(TRIM(RAW.ADDR_CTRY))
QUALIFY SUM(1) OVER (
           PARTITION BY RAW.GROUP_ID
           ORDER BY CASE WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) = 'PRIMARY NAME' THEN 1 ELSE 2 END,
                    CASE WHEN TRIM(RAW.LAST_UPDATED) LIKE '__/__/____'
                         THEN SUBSTR(TRIM(RAW.LAST_UPDATED), 7, 4) ||
                              SUBSTR(TRIM(RAW.LAST_UPDATED), 4, 2) ||
                              SUBSTR(TRIM(RAW.LAST_UPDATED), 1, 2)
                         ELSE '00000000'
                    END DESC,
                    RAW.NAME_6
           ROWS UNBOUNDED PRECEDING
       ) = 1;

/**************************************************************************\
* MAP NATIONALITY VIA REGIME FOR UNMAPPED STATE BODIES                     *
\**************************************************************************/

-- Converted Teradata UPDATE FROM to MERGE INTO.
MERGE INTO DB_TRANS_IMPORTS.SANC_ENTITY_STG AS STG
USING DB_REFERENCE.SANC_REGIME_CTRY AS RGM
ON  STG.REGIME_NM = RGM.REGIME_NM
AND RGM.COMPREHENSIVE_IND = 'Y'
AND STG.ADDR_CTRY_CD IS NULL
AND STG.GROUP_TYP_CD = 'E'
WHEN MATCHED THEN UPDATE SET ADDR_CTRY_CD = RGM.CTRY_2_CD;

/**************************************************************************\
* POPULATE SANC_NAME_STG                                                   *
\**************************************************************************/

DELETE FROM DB_TRANS_IMPORTS.SANC_NAME_STG;

-- Converted OREPLACE/OTRANSLATE and isolated aliases before using them in a window specification.
INSERT INTO DB_TRANS_IMPORTS.SANC_NAME_STG
(
       GROUP_ID,
       NAME_SEQ_NR,
       FULL_NM,
       NORM_NM,
       NON_LATIN_NM,
       ALIAS_TYP_CD,
       ALIAS_QUALITY_CD
)
SELECT NMS.GROUP_ID,
       ROW_NUMBER() OVER (
         PARTITION BY NMS.GROUP_ID
         ORDER BY NMS.ALIAS_RANK_NR, NMS.FULL_NM
       ),
       NMS.FULL_NM,
       TRIM(REPLACE(REPLACE(TRANSLATE(UPPER(TRIM(NMS.FULL_NM)), '.,''-&/()', '        '), '  ', ' '), '  ', ' ')),
       NMS.NON_LATIN_NM,
       NMS.ALIAS_TYP_CD,
       NMS.ALIAS_QUALITY_CD
FROM (
  SELECT BASE.*
  FROM (
    SELECT CAST(TRIM(RAW.GROUP_ID) AS INT) AS GROUP_ID,
           REPLACE(REPLACE(TRIM(
             TRIM(COALESCE(RAW.NAME_1, '')) || ' ' ||
             TRIM(COALESCE(RAW.NAME_2, '')) || ' ' ||
             TRIM(COALESCE(RAW.NAME_3, '')) || ' ' ||
             TRIM(COALESCE(RAW.NAME_4, '')) || ' ' ||
             TRIM(COALESCE(RAW.NAME_5, '')) || ' ' ||
             TRIM(COALESCE(RAW.NAME_6, ''))
           ), '  ', ' '), '  ', ' ') AS FULL_NM,
           NULLIF(TRIM(RAW.NAME_NON_LATIN), '') AS NON_LATIN_NM,
           CASE WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) = 'PRIMARY NAME' THEN 'PRI'
                WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) LIKE 'PRIMARY NAME VARIATION%' THEN 'VAR'
                WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) LIKE '%VARIATION%' THEN 'VAR'
                ELSE 'AKA'
           END AS ALIAS_TYP_CD,
           CASE UPPER(TRIM(RAW.ALIAS_QUALITY))
                WHEN 'GOOD' THEN 'G'
                WHEN 'LOW'  THEN 'L'
                ELSE 'G'
           END AS ALIAS_QUALITY_CD,
           CASE WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) = 'PRIMARY NAME' THEN 1
                WHEN UPPER(TRIM(RAW.ALIAS_TYPE)) LIKE '%VARIATION%' THEN 2
                ELSE 3
           END AS ALIAS_RANK_NR
    FROM DB_TRANS_IMPORTS.AML_SANCSYS_RAW RAW
  ) BASE
  QUALIFY SUM(1) OVER (
             PARTITION BY BASE.GROUP_ID, BASE.FULL_NM
             ORDER BY BASE.ALIAS_RANK_NR
             ROWS UNBOUNDED PRECEDING
           ) = 1
) NMS
WHERE NMS.FULL_NM <> '';

/**************************************************************************\
* TOKENISE THE NORMALISED NAMES                                            *
\**************************************************************************/

-- Converted a Teradata volatile table to a session temporary table; stripped PI, character-set, and commit clauses.
CREATE TEMP TABLE VT_SANC_NM_KEY
(
       NM_KEY   BIGINT NOT NULL,
       NORM_NM  VARCHAR(400) COLLATE UTF8_LCASE
);

INSERT INTO VT_SANC_NM_KEY
SELECT CAST(GROUP_ID AS BIGINT) * 1000 + NAME_SEQ_NR,
       NORM_NM
FROM DB_TRANS_IMPORTS.SANC_NAME_STG
WHERE NORM_NM IS NOT NULL;

-- Converted a Teradata volatile table to a session temporary table; stripped PI, character-set, and commit clauses.
CREATE TEMP TABLE VT_SANC_TOKEN_RAW
(
       NM_KEY        BIGINT NOT NULL,
       TOKEN_SEQ_NR  INT,
       TOKEN_TXT     VARCHAR(400) COLLATE UTF8_LCASE
);

-- Converted STRTOK_SPLIT_TO_TABLE to filtered POSEXPLODE and restored one-based token numbering.
INSERT INTO VT_SANC_TOKEN_RAW
SELECT K.NM_KEY,
       TOK.POS + 1,
       TOK.TOKEN_TXT
FROM VT_SANC_NM_KEY K
LATERAL VIEW POSEXPLODE(
  FILTER(SPLIT(K.NORM_NM, ' '), X -> LENGTH(X) > 0)
) TOK AS POS, TOKEN_TXT;

DELETE FROM DB_TRANS_IMPORTS.SANC_TOKEN_STG;

-- TODO: manual review required — source mode is unknown; VARCHAR(60) overflow is conservatively rejected rather than silently truncated.
-- assumption: values over 60 characters should fail rather than be truncated when source session mode is unknown.
-- Converted integer division and MOD operator semantics explicitly.
INSERT INTO DB_TRANS_IMPORTS.SANC_TOKEN_STG
(
       GROUP_ID,
       NAME_SEQ_NR,
       TOKEN_SEQ_NR,
       TOKEN_TXT
)
SELECT CAST(TRW.NM_KEY DIV 1000 AS INT),
       CAST(MOD(TRW.NM_KEY, 1000) AS SMALLINT),
       CAST(TRW.TOKEN_SEQ_NR AS SMALLINT),
       IF(LENGTH(TRIM(TRW.TOKEN_TXT)) <= 60,
          TRIM(TRW.TOKEN_TXT),
          RAISE_ERROR('TOKEN_TXT exceeds VARCHAR(60)'))
FROM VT_SANC_TOKEN_RAW TRW
WHERE LENGTH(TRIM(TRW.TOKEN_TXT)) >= 2
  AND TRIM(TRW.TOKEN_TXT) NOT IN (
    SELECT NOISE_WORD FROM DB_REFERENCE.NAME_NOISE_WORD
  );

/**************************************************************************\
* SANCTIONS ENTITY: DETERMINE CHANGES                                      *
\**************************************************************************/

-- Converted a Teradata volatile table to a session temporary table and preserved case-insensitive CHAR collation.
CREATE TEMP TABLE VT_SANC_CHG
(
       GROUP_ID    INT NOT NULL,
       CHG_TYP_CD  CHAR(1) COLLATE UTF8_LCASE NOT NULL
);

INSERT INTO VT_SANC_CHG
(
       GROUP_ID,
       CHG_TYP_CD
)
SELECT STG.GROUP_ID,
       CASE WHEN CUR.GROUP_ID IS NULL THEN 'N' ELSE 'U' END
FROM DB_TRANS_IMPORTS.SANC_ENTITY_STG STG
LEFT JOIN DB_ODS.SANC_ENTITY CUR
  ON CUR.GROUP_ID = STG.GROUP_ID
WHERE CUR.GROUP_ID IS NULL
   OR CUR.ACTIVE_IND = 'N'
   OR COALESCE(CUR.GROUP_TYP_CD, '~') <> COALESCE(STG.GROUP_TYP_CD, '~')
   OR COALESCE(CUR.REGIME_NM, '~') <> COALESCE(STG.REGIME_NM, '~')
   OR COALESCE(CUR.LISTED_DT, DATE '0001-01-01') <> COALESCE(STG.LISTED_DT, DATE '0001-01-01')
   OR COALESCE(CUR.LAST_UPD_DT, DATE '0001-01-01') <> COALESCE(STG.LAST_UPD_DT, DATE '0001-01-01')
   OR COALESCE(CUR.UK_SANC_REF, '~') <> COALESCE(STG.UK_SANC_REF, '~')
   OR COALESCE(CUR.BIRTH_DT, DATE '0001-01-01') <> COALESCE(STG.BIRTH_DT, DATE '0001-01-01')
   OR COALESCE(CUR.NATIONALITY_CTRY_CD, '~') <> COALESCE(STG.NATIONALITY_CTRY_CD, '~')
   OR COALESCE(CUR.ADDR_CTRY_CD, '~') <> COALESCE(STG.ADDR_CTRY_CD, '~')
UNION ALL
SELECT CUR.GROUP_ID,
       'D'
FROM DB_ODS.SANC_ENTITY CUR
WHERE CUR.ACTIVE_IND = 'Y'
  AND CUR.GROUP_ID NOT IN (
    SELECT GROUP_ID FROM DB_TRANS_IMPORTS.SANC_ENTITY_STG
  );

/**************************************************************************\
* SANCTIONS ENTITY: CLOSE THE OPEN HISTORY VERSION                         *
\**************************************************************************/

-- Converted Teradata UPDATE FROM to MERGE INTO and TIMESTAMP(0) to TIMESTAMP_NTZ.
MERGE INTO DB_ODS.SANC_ENTITY_HIST AS HST
USING (
  SELECT CHG.GROUP_ID, TTS.TMS
  FROM VT_SANC_CHG CHG
  CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS
) AS SRC
ON HST.GROUP_ID = SRC.GROUP_ID
AND HST.END_DT IS NULL
WHEN MATCHED THEN UPDATE SET
  END_DT = CASE WHEN HST.START_DT < CAST(SRC.TMS AS DATE)
                THEN CAST(SRC.TMS AS DATE) - 1
                ELSE CAST(SRC.TMS AS DATE)
           END,
  CO_UPD_TMS = CAST(SRC.TMS AS TIMESTAMP_NTZ);

/**************************************************************************\
* SANCTIONS ENTITY: WRITE HISTORY VERSIONS                                 *
\**************************************************************************/

-- Converted TIMESTAMP(0) expression casts to timezone-naive TIMESTAMP_NTZ.
INSERT INTO DB_ODS.SANC_ENTITY_HIST
(
       GROUP_ID,
       START_DT,
       UPD_TMS,
       END_DT,
       GROUP_TYP_CD,
       REGIME_NM,
       LISTED_DT,
       LAST_UPD_DT,
       UK_SANC_REF,
       BIRTH_DT,
       NATIONALITY_CTRY_CD,
       ADDR_CTRY_CD,
       ACTION_CD,
       CO_UPD_TMS
)
SELECT STG.GROUP_ID,
       CAST(TTS.TMS AS DATE),
       CAST(TTS.TMS AS TIMESTAMP_NTZ),
       NULL,
       STG.GROUP_TYP_CD,
       STG.REGIME_NM,
       STG.LISTED_DT,
       STG.LAST_UPD_DT,
       STG.UK_SANC_REF,
       STG.BIRTH_DT,
       STG.NATIONALITY_CTRY_CD,
       STG.ADDR_CTRY_CD,
       CHG.CHG_TYP_CD,
       CAST(TTS.TMS AS TIMESTAMP_NTZ)
FROM DB_TRANS_IMPORTS.SANC_ENTITY_STG STG
JOIN VT_SANC_CHG CHG
  ON CHG.GROUP_ID = STG.GROUP_ID
 AND CHG.CHG_TYP_CD IN ('N', 'U')
CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS
UNION ALL
SELECT CUR.GROUP_ID,
       CAST(TTS.TMS AS DATE),
       CAST(TTS.TMS AS TIMESTAMP_NTZ),
       CAST(TTS.TMS AS DATE),
       CUR.GROUP_TYP_CD,
       CUR.REGIME_NM,
       CUR.LISTED_DT,
       CUR.LAST_UPD_DT,
       CUR.UK_SANC_REF,
       CUR.BIRTH_DT,
       CUR.NATIONALITY_CTRY_CD,
       CUR.ADDR_CTRY_CD,
       'D',
       CAST(TTS.TMS AS TIMESTAMP_NTZ)
FROM DB_ODS.SANC_ENTITY CUR
JOIN VT_SANC_CHG CHG
  ON CHG.GROUP_ID = CUR.GROUP_ID
 AND CHG.CHG_TYP_CD = 'D'
CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS;

/**************************************************************************\
* SANCTIONS ENTITY: APPLY TO THE CURRENT TABLE                             *
\**************************************************************************/

-- Converted BT/ET to an atomic Databricks SQL block.
BEGIN ATOMIC
  DELETE FROM DB_ODS.SANC_ENTITY
  WHERE GROUP_ID IN (
    SELECT GROUP_ID FROM VT_SANC_CHG WHERE CHG_TYP_CD IN ('N', 'U')
  );

  INSERT INTO DB_ODS.SANC_ENTITY
  (
         GROUP_ID,
         GROUP_TYP_CD,
         REGIME_NM,
         LISTED_DT,
         LAST_UPD_DT,
         UK_SANC_REF,
         BIRTH_DT,
         NATIONALITY_CTRY_CD,
         ADDR_CTRY_CD,
         PASSPORT_NR,
         ACTIVE_IND,
         DELISTED_DT,
         CO_UPD_TMS
  )
  SELECT STG.GROUP_ID,
         STG.GROUP_TYP_CD,
         STG.REGIME_NM,
         STG.LISTED_DT,
         STG.LAST_UPD_DT,
         STG.UK_SANC_REF,
         STG.BIRTH_DT,
         STG.NATIONALITY_CTRY_CD,
         STG.ADDR_CTRY_CD,
         STG.PASSPORT_NR,
         'Y',
         NULL,
         CAST(TTS.TMS AS TIMESTAMP_NTZ)
  FROM DB_TRANS_IMPORTS.SANC_ENTITY_STG STG
  JOIN VT_SANC_CHG CHG
    ON CHG.GROUP_ID = STG.GROUP_ID
   AND CHG.CHG_TYP_CD IN ('N', 'U')
  CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS;

  -- Converted Teradata UPDATE FROM to MERGE INTO.
  MERGE INTO DB_ODS.SANC_ENTITY AS ENT
  USING (
    SELECT CHG.GROUP_ID, TTS.TMS
    FROM VT_SANC_CHG CHG
    CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS
    WHERE CHG.CHG_TYP_CD = 'D'
  ) AS SRC
  ON ENT.GROUP_ID = SRC.GROUP_ID
  WHEN MATCHED THEN UPDATE SET
    ACTIVE_IND = 'N',
    DELISTED_DT = CAST(SRC.TMS AS DATE),
    CO_UPD_TMS = CAST(SRC.TMS AS TIMESTAMP_NTZ);
END;

/**************************************************************************\
* RELOAD SANC_NAME FOR ACTIVE DESIGNATIONS                                 *
\**************************************************************************/

-- Converted BT/ET to an atomic Databricks SQL block.
BEGIN ATOMIC
  DELETE FROM DB_ODS.SANC_NAME;

  INSERT INTO DB_ODS.SANC_NAME
  (
         GROUP_ID,
         NAME_SEQ_NR,
         FULL_NM,
         NORM_NM,
         NON_LATIN_NM,
         ALIAS_TYP_CD,
         ALIAS_QUALITY_CD,
         TOKEN_CNT,
         CO_UPD_TMS
  )
  SELECT NMS.GROUP_ID,
         NMS.NAME_SEQ_NR,
         NMS.FULL_NM,
         NMS.NORM_NM,
         NMS.NON_LATIN_NM,
         NMS.ALIAS_TYP_CD,
         NMS.ALIAS_QUALITY_CD,
         COALESCE(TKC.TOKEN_CNT, 0),
         CAST(TTS.TMS AS TIMESTAMP_NTZ)
  FROM DB_TRANS_IMPORTS.SANC_NAME_STG NMS
  JOIN DB_ODS.SANC_ENTITY ENT
    ON ENT.GROUP_ID = NMS.GROUP_ID
   AND ENT.ACTIVE_IND = 'Y'
  LEFT JOIN (
    SELECT GROUP_ID,
           NAME_SEQ_NR,
           COUNT(DISTINCT TOKEN_TXT) AS TOKEN_CNT
    FROM DB_TRANS_IMPORTS.SANC_TOKEN_STG
    GROUP BY GROUP_ID, NAME_SEQ_NR
  ) TKC
    ON TKC.GROUP_ID = NMS.GROUP_ID
   AND TKC.NAME_SEQ_NR = NMS.NAME_SEQ_NR
  CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS;
END;

/**************************************************************************\
* RELOAD SANC_NAME_TOKEN FOR ACTIVE DESIGNATIONS                           *
\**************************************************************************/

-- Converted BT/ET to an atomic Databricks SQL block.
BEGIN ATOMIC
  DELETE FROM DB_ODS.SANC_NAME_TOKEN;

  INSERT INTO DB_ODS.SANC_NAME_TOKEN
  (
         TOKEN_TXT,
         GROUP_ID,
         NAME_SEQ_NR,
         TOKEN_SEQ_NR,
         CO_UPD_TMS
  )
  SELECT TOK.TOKEN_TXT,
         TOK.GROUP_ID,
         TOK.NAME_SEQ_NR,
         TOK.TOKEN_SEQ_NR,
         CAST(TTS.TMS AS TIMESTAMP_NTZ)
  FROM DB_TRANS_IMPORTS.SANC_TOKEN_STG TOK
  JOIN DB_ODS.SANC_NAME NMS
    ON NMS.GROUP_ID = TOK.GROUP_ID
   AND NMS.NAME_SEQ_NR = TOK.NAME_SEQ_NR
  CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS
  WHERE NMS.ALIAS_QUALITY_CD <> 'L';
END;

/**************************************************************************\
* EXTRACT SANCTIONED BIC CODES                                             *
\**************************************************************************/

DELETE FROM DB_ODS.SANC_BIC;

-- TODO: manual review required — confirm that fixed-width CHAR(11) blank padding is required by downstream BIC consumers.
-- Converted Teradata REGEXP_SUBSTR/REGEXP_REPLACE extraction and INDEX to Databricks REGEXP_EXTRACT and INSTR.
INSERT INTO DB_ODS.SANC_BIC
(
       BIC_CD,
       GROUP_ID,
       CO_UPD_TMS
)
SELECT DISTINCT
       BIC.BIC_CD,
       BIC.GROUP_ID,
       CAST(TTS.TMS AS TIMESTAMP_NTZ)
FROM (
  SELECT CAST(TRIM(RAW.GROUP_ID) AS INT) AS GROUP_ID,
         CAST(
           NULLIF(
             REGEXP_EXTRACT(
               UPPER(RAW.OTHER_INFO),
               '(?:SWIFT/BIC|SWIFT|BIC)[ :/]{1,3}([A-Z]{6}[A-Z0-9]{2}(?:[A-Z0-9]{3})?)',
               1
             ),
             ''
           ) AS CHAR(11)
         ) AS BIC_CD
  FROM DB_TRANS_IMPORTS.AML_SANCSYS_RAW RAW
  WHERE INSTR(UPPER(RAW.OTHER_INFO), 'BIC') > 0
     OR INSTR(UPPER(RAW.OTHER_INFO), 'SWIFT') > 0
) BIC
JOIN DB_ODS.SANC_ENTITY ENT
  ON ENT.GROUP_ID = BIC.GROUP_ID
 AND ENT.ACTIVE_IND = 'Y'
CROSS JOIN DB_TRANS_IMPORTS.AML_TMS_SANCSYS TTS
WHERE BIC.BIC_CD IS NOT NULL
  AND TRIM(BIC.BIC_CD) <> '';

/**************************************************************************\
* REPORT DESIGNATION MOVEMENTS                                             *
\**************************************************************************/

SELECT CHG_TYP_CD,
       COUNT(*)
FROM VT_SANC_CHG
GROUP BY CHG_TYP_CD
ORDER BY CHG_TYP_CD;

/**************************************************************************\
* DROP TEMPORARY TABLES                                                     *
\**************************************************************************/

-- Converted volatile-table cleanup to temporary-table cleanup.
DROP TEMPORARY TABLE VT_SANC_NM_KEY;
-- Converted volatile-table cleanup to temporary-table cleanup.
DROP TEMPORARY TABLE VT_SANC_TOKEN_RAW;
-- Converted volatile-table cleanup to temporary-table cleanup.
DROP TEMPORARY TABLE VT_SANC_CHG;
